# Model evaluation to run0-spinup_steadystate

This example shows how to load model run0-spinup_steadystate data

In [ ]:
# # skip this if package has already been installed
# !pip install modvis
#import modvis.ats_xdmf as xdmf
#from modvis import colors

import ats_xdmf as xdmf

In [ ]:
# from modvis import ATSutils
# from modvis import utils
# from modvis import general_plots as gp

import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
import h5py

model_dir = "./NF01"
cv_key = 'surface-cell_volume'

generate_plots = False

## Load model data

This will load the `water_balance.dat` file generated from ATS model. The data file includes watershed variables including outlet discharge, ET, and etc. By default, setting `plot=True` will show the water balance plots for `global, canopy, snow, surface, and subsurface` domains. **It is important to check if `max error` is close to zero!** Otherwise, there may be a water balance issue in the model.

In [ ]:
data_file_wb = os.path.join(model_dir, 'water_balance.dat')
data_wb = pd.read_csv(data_file_wb, comment='#')

In [ ]:
data_wb
# {right, left, bottom, top, front, back} face flux all have the same unit [mol d^-1]

In [ ]:
# the goal of vis/obs_file_time_interval here is to make comparable plot
# between results from different time interval setting
vis_file_time_interval = 1 # unit is day. convert the unit if in xml it's not configured as day
obs_file_time_interval = 1

# [to do] automatically determine time_interval from data_wb['time [d]']
# # Calculate cumulative flux by integrating over time
# # Convert time from days to seconds for integration
# time_days = data_wb['time [d]']
# dt_days = np.diff(time_days)
# dt_days = np.append(dt_days, dt_days[-1])  # Add last timestep

In [ ]:
# convert precipitation unit
# water_density = 1000  # kg/m³
# water_molar_mass = 0.018  # kg/mol
rho_m1 = 55500. # moles/m^3, water molar density
rho_m2 = 55000.
surface_area1 = 680 * 1  # m²; y=1 is determined in m2 = watershed_workflow.mesh.Mesh2D.from_Transect(x,z)

with h5py.File(os.path.join(model_dir, 'ats_vis_surface_data.h5'),'r') as d:
    a_key = list(d[cv_key].keys())[0]
    surface_area2 = d[cv_key][a_key][:].sum() # m^2
print(surface_area2)

# Conversion from m/(time interval)
precipitation_m_per_d = data_wb['precipitation [m per time interval]'] / obs_file_time_interval
precipitation_mol_per_d = data_wb['precipitation [m per time interval]'] * 55000. * surface_area2 / obs_file_time_interval

In [ ]:
# water_flux_left_face_m_per_d and water_flux_right_face_m_per_d
# unit m/d, interpretation: amount of water into the domain, in meter, assuming per surface_area2.
# in another word, viewing all flux (unit m/d) as influx/efflux of surface
water_flux_left_face_m_per_d = data_wb['left face flux [mol per time interval]'] / rho_m2 / surface_area2 / obs_file_time_interval
water_flux_right_face_m_per_d = data_wb['right face flux [mol per time interval]'] / rho_m2 / surface_area2 / obs_file_time_interval

# unit mol/d
# [note] vars are independent to surface area in this unit
water_flux_left_face_mol_per_d = data_wb['left face flux [mol per time interval]'] / obs_file_time_interval
water_flux_right_face_mol_per_d = data_wb['right face flux [mol per time interval]'] / obs_file_time_interval

In [ ]:
# water balance check
# noticing error here also depends on time step

# the water storage change
total_water_initial = (data_wb['surface water content [mol]'][0] + 
                      data_wb['subsurface water content [mol]'][0])
storage_change = ((data_wb['surface water content [mol]'] + 
                 data_wb['subsurface water content [mol]']) - total_water_initial) / rho_m2 / surface_area2

# the net flux (precipitation - runoff)
runoff_m_per_d = data_wb['runoff generation [mol per time interval]'] / rho_m2 / surface_area2 / obs_file_time_interval
runoff_mol_per_d = data_wb['runoff generation [mol per time interval]'] / obs_file_time_interval
net_flux = precipitation_m_per_d - runoff_m_per_d - water_flux_left_face_m_per_d - water_flux_right_face_m_per_d
net_flux_mol_per_d = precipitation_mol_per_d - runoff_mol_per_d - water_flux_left_face_mol_per_d - water_flux_right_face_mol_per_d

#cumulative_water -> changes of water amount during 1 time interval
cumulative_flux = np.cumsum(net_flux)*obs_file_time_interval


In [ ]:
# print(storage_change)
# print(cumulative_flux)

In [ ]:
print(precipitation_mol_per_d)
print(runoff_mol_per_d)
print(water_flux_left_face_mol_per_d)
print(water_flux_right_face_mol_per_d)
print(net_flux_mol_per_d)

In [ ]:
print(precipitation_mol_per_d.iloc[-1] - 
      water_flux_left_face_mol_per_d.iloc[-1] -
      runoff_mol_per_d.iloc[-1] -
      water_flux_right_face_mol_per_d.iloc[-1])

In [ ]:
# Create the figure with subplots
fig, ax = plt.subplots(2, 3, figsize=(12, 6))

# Plot 1: Subsurface Water Content
ax[0,0].plot(data_wb['time [d]'], data_wb['subsurface water content [mol]']/ rho_m2 / surface_area2, 
         label='Subsurface Water Content', linewidth=2)
ax[0,0].set_xlabel('Time [d]')
ax[0,0].set_ylabel('Subsurface Water Content [m]')
ax[0,0].set_title('Subsurface Water Content')
ax[0,0].grid(True, linestyle='--', alpha=0.7)
ax[0,0].legend()
ax[0,0].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# Plot 2: Surface Water Content
ax[0,1].plot(data_wb['time [d]'], data_wb['surface water content [mol]'] / rho_m2 / surface_area2, 
         label='Surface Water Content', linewidth=2)
ax[0,1].set_xlabel('Time [d]')
ax[0,1].set_ylabel('Surface Water Content [m]')
ax[0,1].set_title('Surface Water Content')
ax[0,1].grid(True, linestyle='--', alpha=0.7)
ax[0,1].legend()
ax[0,1].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# Plot 3: Precipitation
# ax[0,2].plot(data_wb['time [d]'], data_wb['precipitation [m d^-1]'], 
#          label='precipitation', linewidth=2)
ax[0,2].plot(data_wb['time [d]'], precipitation_m_per_d, 
         label='precipitation', linewidth=2)
ax[0,2].set_xlabel('Time [d]')
ax[0,2].set_ylabel('precipitation [m d^-1]')
ax[0,2].set_title('precipitation')
ax[0,2].grid(True, linestyle='--', alpha=0.7)
ax[0,2].legend()
ax[0,2].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# Plot 4: runoff generation
ax[1,0].plot(data_wb['time [d]'], runoff_m_per_d, 
         label='runoff generation', linewidth=2)
ax[1,0].set_xlabel('Time [d]')
ax[1,0].set_ylabel('runoff generation [m d^-1]')
ax[1,0].set_title('runoff generation')
ax[1,0].grid(True, linestyle='--', alpha=0.7)
ax[1,0].legend()
ax[1,0].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# Plot 3: Groundwater Table
data_file_gt = os.path.join(model_dir, 'groundwater_table.dat')
data_gt = pd.read_csv(data_file_gt, comment='#')
ax[1,1].plot(data_gt['time [d]'], data_gt['groundwater table'], 
         label='Groundwater table', linewidth=2)
ax[1,1].set_xlabel('Time [d]')
ax[1,1].set_ylabel('Groundwater table [m]')
ax[1,1].set_title('Groundwater Table')
ax[1,1].grid(True, linestyle='--', alpha=0.7)
ax[1,1].legend()
ax[1,1].ticklabel_format(style='sci', axis='y', scilimits=(0,0))


ax[1,2].plot(data_wb['time [d]'], storage_change, 
         label='Storage Change', linewidth=2)
ax[1,2].plot(data_wb['time [d]'], cumulative_flux, 
         label='Cumulative Net Flux', linewidth=2)
ax[1,2].plot(data_wb['time [d]'], storage_change - cumulative_flux, 
         label='Water Balance Error', linewidth=2, linestyle='--')
ax[1,2].set_xlabel('Time [d]')
ax[1,2].set_ylabel('Water [m]')
ax[1,2].set_title('Water Balance')
ax[1,2].grid(True, linestyle='--', alpha=0.7)
ax[1,2].legend()
ax[1,2].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

# Adjust layout
plt.tight_layout()
plt.show()

# read visualization file

In [ ]:
# for [subsurface right face], compare darcy_velocity from visualization file and water_flux from observation file
# for [surface right face], it's not comparable. I don't understand surface-velocity.

# based on ATS online documentation, unit of darcy_velocity is m/s, while unit of water_flux is mol/s
# unit conversion: darcy_velocity*surface_area*55000 mol/m3
# noticing: observation file is doing [extensive integral] of water_flux

In [ ]:
xlim_preset = (-10,690)
ylim_preset = (940,1120)

In [ ]:
vis_surf = xdmf.VisFile(directory=model_dir,
                        domain="surface",
                        time_unit='d')
#select same output as subsurface
#vis_surf.filterIndices(steps)
vis_surf.loadMesh(order=['x','z'])

vis = xdmf.VisFile(directory=model_dir, 
                   time_unit='d')
#select output every 2 days
#vis.filterIndices(steps)
vis.loadMeshPolygons()

In [ ]:
# Plot
fig, ax = plt.subplots(1,1,figsize=(10,2.6),sharex=True)
# norm = mcolors.TwoSlopeNorm(vmin=-16, vcenter=-15, vmax=-11.5)
sat = vis.get("relative_permeability", vis.cycles[0]); sat = np.log10(sat)
poly = vis.getMeshPolygons(cmap='cividis', linewidth=0.1, edgecolor='k')
# sat = vis.get("saturation_liquid", vis.cycles[0]); sat[:] = 0
# poly = vis.getMeshPolygons(cmap='Blues_r', linewidth=0.1, edgecolor='k', norm=norm)
poly.set_array(sat)
#poly.set_clim(-16,-11.5)
ax.add_collection(poly)

elev = vis_surf.get('surface-elevation', vis.cycles[-1])
depth = vis_surf.get('surface-ponded_depth', vis.cycles[-1])

ax.plot(vis_surf.centroids[:,0], elev+depth, 'white', linewidth=2, alpha=0)
# ax.axhline(y=0)
plt.colorbar(poly,shrink=1., ax=ax, label = r'$\text{log} \ \text{Permeability} \ (m^2)$')
# ax.plot(0,0)
ax.set(xlabel='Distance (m)', ylabel='Elevation (m)',ylim=ylim_preset, xlim=xlim_preset)

if generate_plots:
    output_filename = '../images/fig1c-hillslope2d.tif'
    fig.savefig(output_filename, dpi=300, pil_kwargs={'compression': 'tiff_lzw'})

In [ ]:
# from full_watershed-workflow
# 18 layers in z
dzs_soil = [0.05, 0.05, 0.05, 0.1, 0.25, 0.5, 0.5, 0.5]
dzs_geo = [1., 1., 1.5, 1.5, 2., 2., 2., 3., 3., 3.]
dzs_merged = np.concatenate((dzs_soil, dzs_geo))

print(dzs_merged)

# since width in y-axis is 1m, so areas = dzs_merged

In [ ]:
# Assuming your x-coordinates increase from left to right
# Find the maximum x-coordinate of the cell centroids
max_x = np.max(vis.centroids[:, 0])

# Identify the indices of the cells whose centroids have an x-coordinate
# very close to the maximum x-coordinate. This will give you the cells
# on the right boundary. You might need to adjust the tolerance
# depending on the precision of your mesh.
tolerance = 1e-6  # Adjust as needed
right_boundary_cell_indices = np.where(np.abs(vis.centroids[:, 0] - max_x) < tolerance)[0]

# Sort the right boundary cell indices based on their z-centroid
# from largest (top) to smallest (bottom)
sorted_indices = right_boundary_cell_indices[np.argsort(vis.centroids[right_boundary_cell_indices, 2])[::-1]]

print("Original right boundary cell indices:", right_boundary_cell_indices)
print("Sorted right boundary cell indices (top to bottom):", sorted_indices)
print(vis.centroids[sorted_indices, 2])

In [ ]:
vis.centroids[sorted_indices, 2]

In [ ]:
varn = "darcy_velocity.0"
all_steps_sum = []

for step in range(len(vis.cycles)):
    zdata = vis.get(varn, vis.cycles[step])

    # Multiply the zdata at the sorted right boundary indices by the merged array
    # Ensure that the lengths are compatible for element-wise multiplication
    if len(zdata[sorted_indices]) == len(dzs_merged):
        multiplied_data = zdata[sorted_indices] * dzs_merged
        step_sum = np.sum(multiplied_data)
        all_steps_sum.append(step_sum)
    else:
        print(f"Warning: Length mismatch between zdata[sorted_indices] ({len(zdata[sorted_indices])}) and merged_array ({len(merged_array)}). Skipping multiplication and summation for this step.")
        all_steps_sum.append(None) # Or some other indicator of failure

print("\nSum of multiplied data for all steps:", all_steps_sum)

all_steps_sum_array = np.array(all_steps_sum)

In [ ]:
# all_steps_sum is in m3/s
# compare with data_wb['right face flux'] in mol/(time interval)

xmin = -10
xmax = 3650

# Create the figure with subplots
fig, ax = plt.subplots(1, 3, figsize=(12, 3))

# Get the time data for coloring
time_data = data_wb['time [d]']

# Plot 1: Subsurface Water Content
ax[0].scatter(data_wb['time [d]'], data_wb['right face flux [mol per time interval]']/obs_file_time_interval, 
           c=time_data, cmap='viridis', s=20)
ax[0].set_xlabel('Time [d]')
ax[0].set_ylabel('water flux [mol/d]')
ax[0].grid(True, linestyle='--', alpha=0.7)
ax[0].legend()
ax[0].ticklabel_format(style='sci', axis='y', scilimits=(0,0))
ax[0].set_xlim(xmin, xmax)
ax[0].set_title('Obs - right face flux')

# Plot 2: Surface Water Content
ax[1].scatter(data_wb['time [d]'], all_steps_sum_array*rho_m2*86400, 
           c=time_data, cmap='viridis', s=20)
ax[1].set_xlabel('Time [d]')
ax[1].set_ylabel('water flux [mol/d]')
ax[1].grid(True, linestyle='--', alpha=0.7)
ax[1].legend()
ax[1].ticklabel_format(style='sci', axis='y', scilimits=(0,0))
ax[1].set_xlim(xmin, xmax)
ax[0].set_title('Vis - darcy flux')

# Plot 3: 1-1 plot
ax[2].scatter(data_wb['right face flux [mol per time interval]']/obs_file_time_interval, all_steps_sum_array*rho_m2*86400,
              c=time_data, cmap='viridis', marker='.', s=50)
ax[2].set_xlabel('Obs - water flux')
ax[2].set_ylabel('Vis - darcy velocity * molar density')
ax[2].legend()
ax[2].ticklabel_format(style='sci', axis='x', scilimits=(0,0))
ax[2].ticklabel_format(style='sci', axis='y', scilimits=(0,0))
x_vals = np.linspace(0, 1.3e5, 100)
ax[2].plot(x_vals, x_vals, 'k--')
ax[2].set_title('Vis - Obs 1-1 Plot')